In [0]:
df = spark.read.parquet("/mnt/bronze/cpt_codes")

In [0]:
display(df)

In [0]:
df.createOrReplaceTempView('cptcodes')

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW quality_checks AS
SELECT  
  cpt_codes,
  procedure_code_category,
  procedure_code_descriptions,
  code_status,
  CASE 
    WHEN cpt_codes IS NULL OR procedure_code_descriptions IS NULL THEN TRUE
    ELSE FALSE
  END AS is_quarantined
FROM cptcodes

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.cptcodes(
  cpt_codes STRING,
  procedure_code_category STRING,
  procedure_code_descriptions STRING,
  code_status STRING,
  is_quarantined BOOLEAN,
  audit_insert_date TIMESTAMP,
  audit_update_date TIMESTAMP,
  is_current BOOLEAN
)
USING DELTA

In [0]:
%sql
MERGE INTO silver.cptcodes AS target
USING quality_checks AS source
ON target.cpt_codes = source.cpt_codes AND target.is_current = TRUE
WHEN MATCHED AND(
  target.procedure_code_category != source.procedure_code_category OR
  target.procedure_code_descriptions != source.procedure_code_descriptions OR
  target.code_status != source.code_status OR
  target.is_quarantined != source.is_quarantined
)
THEN UPDATE SET
  target.is_current = false,
  target.audit_update_date = current_timestamp()


In [0]:
%sql
MERGE INTO silver.cptcodes AS target
USING quality_checks AS source
ON target.cpt_codes = source.cpt_codes AND target.is_current = TRUE
WHEN NOT MATCHED THEN
INSERT (
  cpt_codes,
  procedure_code_category,
  procedure_code_descriptions,
  code_status,
  is_quarantined,
  audit_insert_date,
  audit_update_date,
  is_current
)
VALUES(
  source.cpt_codes,
  source.procedure_code_category,
  source.procedure_code_descriptions,
  source.code_status,
  source.is_quarantined,
  current_timestamp(),
  current_timestamp(),
  true
)

In [0]:
%sql
SELECT * FROM silver.cptcodes